<a href="https://colab.research.google.com/github/akshat280706/ML-Lab-Experiment/blob/main/241080009_akshat_ML_LAB2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import files
uploaded=files.upload()

Saving mushrooms.csv to mushrooms.csv


In [5]:
import pandas as pd
import math
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import(
    accuracy_score, confusion_matrix, precision_score,
    recall_score, f1_score
)

#to read the dataset
data=pd.read_csv("mushrooms.csv")
target="class"

In [ ]:
def entropy(data):
  counts=data[target].value_counts()
  total=len(data)
  e=0
  for count in counts:
    p=count/total
    e-=p*math.log2(p)
  return e


def information_gain(data, attribute):
  total_entropy=entropy(data)
  weighted_entropy=0

  for value in data[attribute].unique():
    subset=data[data[attribute]==value]
    weight=len(subset)/len(data)
    weighted_entropy+=weight*entropy(subset)
  return total_entropy-weighted_entropy


def ID3(data,attributes):
  classes=data[target]

  if len(classes.unique())==1:
    return classes.iloc[0] #i reached leaf node

  if len(attributes)==0:
    return classes.mode()[0]

  print("\ncurrent entropy: ", round(entropy(data),5))
  print("\ninformation gain: ")

  best_attribute=None
  best_gain=-1

  for attribute in attributes:
    current_gain=information_gain(data,attribute)
    print(attribute,"=",round(current_gain,5))

    if(current_gain>best_gain):
      best_gain=current_gain
      best_attribute=attribute
  print("selected attribute: ", best_attribute)
  tree={best_attribute:{}}

  remaining_attribute=[
      attribute
      for attribute in attributes
      if attribute!=best_attribute
  ]

  for value in data[best_attribute].unique():
    subset=data[data[best_attribute]==value]

    if len(subset)==0:
      tree[best_attribute][value]=classes.mode()[0]
    else:
      subset=subset.drop(columns=[best_attribute])

      tree[best_attribute][value]=ID3(
          subset, remaining_attribute
      )
  return tree

def predict(tree, row):
    if not isinstance(tree, dict):
        return tree

    root = list(tree.keys())[0]
    value = row[root]

    if value not in tree[root]:
        return None
    return predict(tree[root][value], row)

def print_rules(tree, rule=""):
  if not isinstance(tree, dict):
    print(rule,"then class= ", tree)
    return
  root=list(tree.keys())[0]
  for value in tree[root]:
    if rule=="":
      new_rule="If "+root+" = "+str(value)
    else:
      new_rule=rule+" And "+root+" = "+str(value)
    print_rules(tree[root][value],new_rule)


train_data, test_data = train_test_split(
    data,
    test_size=0.2,
    random_state=42,
    stratify=data[target]
)
attributes = list(train_data.columns)
attributes.remove(target)

tree = ID3(train_data, attributes)
print("\nRoot node: ")
print(list(tree.keys())[0])
print("\nDecision rules: ")
print_rules(tree)

predictions = []
for _, row in test_data.iterrows():
    predictions.append(predict(tree, row))

correct = sum(
    actual == predicted
    for actual, predicted in zip(
        test_data[target],
        predictions
    )
)

manual_accuracy = correct / len(test_data)
print("\nID3 accuracy(manual):")
print(round(manual_accuracy * 100, 2), "%")


encoded = data.copy()

for column in encoded.columns:
    encoded[column] = LabelEncoder().fit_transform(
        encoded[column]
    )

X = encoded.drop(target, axis=1)
Y = encoded[target]

X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=42,
    stratify=Y
)

model = DecisionTreeClassifier(
    criterion="entropy",
    random_state=42
)

model.fit(X_train, Y_train)

prediction = model.predict(X_test)

print("\nScikit-learn accuracy:")
print(round(accuracy_score(Y_test, prediction) * 100, 2), "%")

print("\nConfusion Matrix")
print(confusion_matrix(Y_test, prediction))
print("\nPrecision")
print(precision_score(Y_test, prediction))
print("\nRecall")
print(recall_score(Y_test, prediction))
print("\nF1 Score")
print(f1_score(Y_test, prediction))

print("\nComparison")
if manual_accuracy > accuracy_score(Y_test, prediction):
    print("Manual ID3 performs better.")
elif manual_accuracy < accuracy_score(Y_test, prediction):
    print("Scikit-learn performs better.")
else:
    print("Both models give similar results.")



current entropy:  0.99907

information gain: 
cap-shape = 0.04837
cap-surface = 0.02648
cap-color = 0.03483
bruises = 0.19325
odor = 0.90531
gill-attachment = 0.01412
gill-spacing = 0.10135
gill-size = 0.22358
gill-color = 0.41618
stalk-shape = 0.00778
stalk-root = 0.1371
stalk-surface-above-ring = 0.28871
stalk-surface-below-ring = 0.27305
stalk-color-above-ring = 0.25596
stalk-color-below-ring = 0.2447
veil-type = 0.0
veil-color = 0.02291
ring-number = 0.03758
ring-type = 0.31931
spore-print-color = 0.48586
population = 0.20385
habitat = 0.15274
selected attribute:  odor

current entropy:  0.21563

information gain: 
cap-shape = 0.04221
cap-surface = 0.01591
cap-color = 0.0939
bruises = 0.00113
gill-attachment = 0.00273
gill-spacing = 0.00648
gill-size = 0.0203
gill-color = 0.08836
stalk-shape = 0.06295
stalk-root = 0.02217
stalk-surface-above-ring = 0.02241
stalk-surface-below-ring = 0.04976
stalk-color-above-ring = 0.03439
stalk-color-below-ring = 0.05968
veil-type = 0.0
veil-colo

In [6]:
print(data["stalk-root"].unique())

['e' 'c' 'b' 'r' '?']
